# Product Recommendations: Baseline Models & MLflow Tracking

This notebook implements and evaluates the three recommendation baselines specified in **Section 6 & 8** of the [Pipeline Implementation Guide](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/documents/pipeline-implementation-guide.md):
1. **Global Popularity**: Recommend products with the highest purchase volume in the past 90 days (with all-time fallback).
2. **Category Popularity**: Recommend top products from categories matching the customer's recent purchases, with global popularity fallback.
3. **Purchase History**: Recommend the customer's previously purchased products ranked by frequency and recency, with global popularity fallback.

---

### MLOps Governance & Data Pipeline Connection
- **Upstream Data Lineage**: Reads directly from the analytical warehouse ([`data/warehouse/olist.duckdb`](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/warehouse/olist.duckdb)) produced by `prepare_curated_data.py`.
- **Chronological Time-Split Snapshots**: Strict cutoffs prevent future data leakage:
  - **Validation Snapshot**: July 1, 2018 (evaluates July 2018 purchases).
  - **Test Snapshot**: August 1, 2018 (evaluates August 2018 purchases).
- **Dedicated MLflow Tracking**: Logs runs under the dedicated experiment `ecommerce-recommendations`, completely separated from forecasting baselines as required by [`.agents/rules/mlflow_experiment_tracking.md`](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/.agents/rules/mlflow_experiment_tracking.md).
- **Cold-Start Cohort Analysis**: Evaluates **Warm** (repeat) and **New** (cold-start) customer cohorts separately.


In [1]:
import os
import sys
import json
import math
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import duckdb
import numpy as np
import pandas as pd
import mlflow
from IPython.display import display

# Identify repository root
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/data/prepare_curated_data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook must be run from inside the repository.")
sys.path.insert(0, str(ROOT))

# Paths from upstream pipeline
DATA_DIR = ROOT / "data"
WAREHOUSE_PATH = DATA_DIR / "warehouse" / "olist.duckdb"
BASELINES_DIR = DATA_DIR / "baselines"
BASELINES_DIR.mkdir(parents=True, exist_ok=True)

if not WAREHOUSE_PATH.is_file():
    raise FileNotFoundError(f"Missing DuckDB warehouse: {WAREHOUSE_PATH}. Run scripts/data/prepare_curated_data.py first.")

# Configure dedicated MLflow tracking
MLFLOW_DB = BASELINES_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
EXPERIMENT_NAME = "ecommerce-recommendations"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Repository Root: {ROOT}")
print(f"Warehouse Path:  {WAREHOUSE_PATH}")
print(f"MLflow Store:    {mlflow.get_tracking_uri()}")
print(f"MLflow Exp:      {EXPERIMENT_NAME}")


Repository Root: /Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender
Warehouse Path:  /Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/warehouse/olist.duckdb
MLflow Store:    sqlite:////Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/data/baselines/mlflow.db
MLflow Exp:      ecommerce-recommendations


2026/10/27 11:32:20 INFO mlflow.tracking.fluent: Experiment with name 'ecommerce-recommendations' does not exist. Creating a new experiment.


## 1. Define Chronological Recommendation Snapshots

To prevent lookahead bias (data leakage), we simulate making recommendations at the start of each month using **only orders delivered before that moment**:

- **Validation Period**:
  - Snapshot: **July 1, 2018**
  - Features: Purchases delivered strictly before July 1, 2018.
  - Evaluation Labels: Purchases made between July 1 and August 1, 2018 that were successfully delivered.
- **Test Period**:
  - Snapshot: **August 1, 2018**
  - Features: Purchases delivered strictly before August 1, 2018.
  - Evaluation Labels: Purchases made between August 1 and September 1, 2018 that were successfully delivered.

Top $K = 10$ recommendations are generated for every customer who made a purchase in the evaluation window.


In [2]:
K = 10
RECENT_DAYS = 90
REC_PERIODS = {
    "validation": (pd.Timestamp("2018-07-01"), pd.Timestamp("2018-08-01")),
    "test": (pd.Timestamp("2018-08-01"), pd.Timestamp("2018-09-01")),
}

def load_history(warehouse: Path, cutoff: pd.Timestamp) -> pd.DataFrame:
    """Load customer interaction history strictly delivered before the cutoff date."""
    with duckdb.connect(str(warehouse), read_only=True) as con:
        return con.execute("""
            SELECT order_id, order_item_id, customer_unique_id, product_id,
                   customer_state,
                   COALESCE(product_category_name_english, 'unknown') AS category,
                   CAST(order_purchase_timestamp AS TIMESTAMP) AS purchased_at
            FROM order_items
            WHERE order_status = 'delivered'
              AND CAST(order_purchase_timestamp AS TIMESTAMP) < ?
              AND CAST(order_delivered_customer_date AS TIMESTAMP) < ?
        """, [cutoff.to_pydatetime(), cutoff.to_pydatetime()]).df()

def load_labels(warehouse: Path, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """Load ground truth purchases made during the evaluation window."""
    with duckdb.connect(str(warehouse), read_only=True) as db:
        return db.execute("""
            SELECT DISTINCT customer_unique_id, product_id
            FROM order_items
            WHERE order_status = 'delivered'
              AND CAST(order_purchase_timestamp AS TIMESTAMP) >= ?
              AND CAST(order_purchase_timestamp AS TIMESTAMP) < ?
            ORDER BY customer_unique_id, product_id
        """, [start.to_pydatetime(), end.to_pydatetime()]).df()

snapshot_df = pd.DataFrame(REC_PERIODS, index=["snapshot_cutoff", "label_end_exclusive"]).T
display(snapshot_df)


           snapshot_cutoff label_end_exclusive
validation      2018-07-01          2018-08-01
test            2018-08-01          2018-09-01


## 2. The Three Recommendation Baselines

We implement three classical e-commerce heuristic recommendation strategies:

| Baseline Model | Recommendation Strategy | Cold-Start (New Customer) Handling |
|---|---|---|
| **`global_popularity`** | Ranks items by total units purchased across all customers in the last 90 days. | Unchanged (same global top list for all users). |
| **`category_popularity`** | Identifies the customer's recently purchased categories, pulls top items in those categories, then fills remaining spots with global popular items. | Falls back to `global_popularity`. |
| **`purchase_history`** | Ranks items the customer previously bought by purchase frequency, breaking ties by recency, then fills remaining spots with global popular items. | Falls back to `global_popularity`. |

All recommendation lists are deduplicated and padded to size $K=10$ using the global popularity ranking.


In [3]:
def unique_top(products: list[str], k: int) -> list[str]:
    return list(dict.fromkeys(products))[:k]

def prepare_recommender(history: pd.DataFrame, cutoff: pd.Timestamp):
    """Precompute indexed candidate lists for global, category, and user history."""
    recent = history.loc[history.purchased_at.ge(cutoff - pd.Timedelta(days=RECENT_DAYS))]
    popular_history = recent if not recent.empty else history
    
    # 1. Global popularity list
    counts = popular_history.groupby("product_id").size().rename("count").reset_index()
    global_order = counts.sort_values(["count", "product_id"], ascending=[False, True]).product_id.tolist()
    
    # 2. Product category index & customer recent categories
    categories = history.drop_duplicates("product_id").set_index("product_id").category.to_dict()
    recent_categories = recent.groupby("customer_unique_id").category.agg(set).to_dict()
    
    # 3. User purchase history (frequency + recency)
    preferences = history.groupby(["customer_unique_id", "product_id"], as_index=False).agg(
        frequency=("order_id", "size"),
        last_purchase=("purchased_at", "max")
    )
    preferences = preferences.sort_values(
        ["customer_unique_id", "frequency", "last_purchase", "product_id"],
        ascending=[True, False, False, True]
    )
    history_orders = preferences.groupby("customer_unique_id").product_id.agg(list).to_dict()
    
    category_lists = {}
    
    def recommend(customer: str, method: str, k: int) -> list[str]:
        if method == "global_popularity":
            selected = global_order
        elif method == "category_popularity":
            key = frozenset(recent_categories.get(customer, set()))
            if key not in category_lists:
                category_lists[key] = [p for p in global_order if categories.get(p) in key][:k]
            selected = category_lists[key] + global_order[:k]
        elif method == "purchase_history":
            selected = history_orders.get(customer, [])[:k] + global_order[:k]
        else:
            raise ValueError(f"Unknown recommendation method: {method}")
        return unique_top(selected, k)
        
    return recommend

# Worked toy example to verify the three rules
toy_history = pd.DataFrame({
    "order_id": ["O1", "O2", "O3", "O3", "O4"],
    "customer_unique_id": ["C1", "C2", "C2", "C2", "C3"],
    "product_id": ["Shoes", "Mixer", "Mixer", "Socks", "Bottle"],
    "category": ["Sports", "Kitchen", "Kitchen", "Sports", "Sports"],
    "purchased_at": pd.to_datetime(["2018-06-01"] * 5),
})
toy_rec = prepare_recommender(toy_history, pd.Timestamp("2018-07-01"))
toy_comparison = pd.DataFrame({m: toy_rec("C1", m, 3) for m in ["global_popularity", "category_popularity", "purchase_history"]})
print("Toy Recommendation Comparison for Customer 'C1' (Past purchase: Shoes):")
display(toy_comparison)


Toy Recommendation Comparison for Customer 'C1' (Past purchase: Shoes):
  global_popularity category_popularity purchase_history
0             Mixer              Bottle            Shoes
1            Bottle               Shoes            Mixer
2             Shoes               Socks           Bottle


## 3. Top-K Ranking Evaluation Metrics

We evaluate recommendation lists using standard information retrieval and ranking metrics at cut-off $K=10$:

- **Precision@K**: Fraction of recommended items that were actually purchased:
  $$\text{Precision@K} = \frac{|\text{Recommendations} \cap \text{Relevant}|}{K}$$
- **Recall@K**: Fraction of all purchased items that were captured in the top-$K$:
  $$\text{Recall@K} = \frac{|\text{Recommendations} \cap \text{Relevant}|}{|\text{Relevant}|}$$
- **Hit Rate@K**: Binary indicator (1 if at least one recommended item was purchased, else 0).
- **NDCG@K (Normalized Discounted Cumulative Gain)**: Accounts for the position of relevant items. A hit at rank 1 receives higher reward than a hit at rank 10:
  $$\text{DCG@K} = \sum_{i=1}^K \frac{\mathbb{I}(\text{hit}_i)}{\log_2(i + 1)}, \quad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$
- **Catalog Coverage**: Proportion of catalog items recommended across all customers.


In [4]:
def ranking_metrics(recommendations: list[str], relevant: set[str], k: int) -> dict[str, float]:
    recommendations = recommendations[:k]
    hits = [int(p in relevant) for p in recommendations]
    dcg = sum(hit / math.log2(position + 2) for position, hit in enumerate(hits))
    ideal = sum(1 / math.log2(position + 2) for position in range(min(k, len(relevant))))
    return {
        "precision_at_k": sum(hits) / k,
        "recall_at_k": sum(hits) / len(relevant) if relevant else 0.0,
        "hit_rate_at_k": float(any(hits)),
        "ndcg_at_k": dcg / ideal if ideal else 0.0,
    }

worked_example = ranking_metrics(["Shoes", "Socks", "Bottle"], {"Socks", "Bag"}, 3)
print("Worked Example Metrics (Recommended: Shoes, Socks, Bottle; Actually Bought: Socks, Bag):")
display(pd.Series(worked_example))


Worked Example Metrics (Recommended: Shoes, Socks, Bottle; Actually Bought: Socks, Bag):
precision_at_k    0.333333
recall_at_k       0.500000
hit_rate_at_k     1.000000
ndcg_at_k         0.386853
dtype: float64


In [5]:
REC_METHODS = ["global_popularity", "category_popularity", "purchase_history"]
recommendation_rows = []
cohort_rows = []
example_recommendations = []

for split, (cutoff, end) in REC_PERIODS.items():
    history = load_history(WAREHOUSE_PATH, cutoff)
    labels = load_labels(WAREHOUSE_PATH, cutoff, end)
    recommend = prepare_recommender(history, cutoff)
    
    truths = labels.groupby("customer_unique_id").product_id.agg(set).to_dict()
    known_customers = set(history.customer_unique_id)
    catalog_ids = set(history.product_id)
    label_catalog_coverage = labels.product_id.isin(catalog_ids).mean()
    
    for method in REC_METHODS:
        records = []
        recommended_catalog = set()
        
        for customer, relevant in truths.items():
            products = recommend(customer, method, K)
            recommended_catalog.update(products)
            is_warm = customer in known_customers
            
            rec_scores = ranking_metrics(products, relevant, K)
            rec_scores["cohort"] = "warm" if is_warm else "new"
            records.append(rec_scores)
            
            if customer == next(iter(truths)):
                example_recommendations.append({
                    "split": split,
                    "baseline": method,
                    "customer_id": customer[:8] + "...",
                    "recommendations": ", ".join([p[:8] + "..." for p in products[:5]]),
                    "actual_purchases": ", ".join([p[:8] + "..." for p in sorted(relevant)])
                })
                
        scores_df = pd.DataFrame(records)
        overall = scores_df.drop(columns="cohort").mean().to_dict()
        
        recommendation_rows.append({
            "split": split,
            "baseline": method,
            **overall,
            "customers": len(scores_df),
            "warm_customers": int(scores_df.cohort.eq("warm").sum()),
            "catalog_coverage": len(recommended_catalog) / len(catalog_ids),
            "label_catalog_coverage": label_catalog_coverage,
        })
        
        for cohort, group in scores_df.groupby("cohort"):
            cohort_rows.append({
                "split": split,
                "baseline": method,
                "cohort": cohort,
                "customers": len(group),
                **group.drop(columns="cohort").mean().to_dict()
            })

recommendation_metrics = pd.DataFrame(recommendation_rows)
recommendation_cohorts = pd.DataFrame(cohort_rows)
example_df = pd.DataFrame(example_recommendations)

print("Overall Recommendation Baseline Performance:")
display(recommendation_metrics)

print("\nCustomer Cohort Breakdown (Warm vs. New):")
display(recommendation_cohorts)

# Save evaluation CSVs
recommendation_metrics.to_csv(BASELINES_DIR / "recommendation_metrics.csv", index=False)
recommendation_cohorts.to_csv(BASELINES_DIR / "recommendation_cohorts.csv", index=False)
example_df.to_csv(BASELINES_DIR / "example_recommendations.csv", index=False)


Overall Recommendation Baseline Performance:
        split             baseline  ...  catalog_coverage  label_catalog_coverage
0  validation    global_popularity  ...          0.000362                0.613826
1  validation  category_popularity  ...          0.008145                0.613826
2  validation     purchase_history  ...          0.006154                0.613826
3        test    global_popularity  ...          0.000339                0.602479
4        test  category_popularity  ...          0.007532                0.602479
5        test     purchase_history  ...          0.006751                0.602479

[6 rows x 10 columns]

Customer Cohort Breakdown (Warm vs. New):
         split             baseline  ... hit_rate_at_k  ndcg_at_k
0   validation    global_popularity  ...      0.012079   0.005752
1   validation    global_popularity  ...      0.007194   0.004411
2   validation  category_popularity  ...      0.012079   0.005752
3   validation  category_popularity  ...      0.043

## 4. MLflow Experiment Tracking

Now we track each recommendation baseline as a distinct run in the dedicated MLflow experiment `ecommerce-recommendations`.

Adhering to [`.agents/rules/mlflow_experiment_tracking.md`](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/.agents/rules/mlflow_experiment_tracking.md):
- **Parameters**: `task: "recommendation"`, `k: 10`, `popularity_window_days: 90`, `inventory_filter: "unavailable_in_olist"`, cutoff dates.
- **Metrics**: `val_ndcg_at_k`, `val_recall_at_k`, `val_precision_at_k`, `val_hit_rate_at_k`, `val_warm_ndcg_at_k`, `val_new_ndcg_at_k`, and their `test_*` equivalents.
- **Tags**: `selected_on_validation`, `model_status: "champion_baseline"`, `pipeline_integrated: "true"`.
- **Artifacts**: `recommendation_metrics.csv`, `recommendation_cohorts.csv`, `example_recommendations.csv`.


In [6]:
val_subset = recommendation_metrics.query("split == 'validation'")
champion_baseline = val_subset.sort_values("ndcg_at_k", ascending=False).iloc[0]["baseline"]
print(f"🏆 Champion Recommendation Baseline selected on Validation NDCG@{K}: '{champion_baseline}'")

mlflow_runs = []
for method in recommendation_metrics["baseline"].unique():
    is_champion = (method == champion_baseline)
    
    with mlflow.start_run(run_name=f"baseline__{method}") as run:
        # 1. Log Parameters
        mlflow.log_params({
            "task": "recommendation",
            "model_family": "heuristic_baseline",
            "baseline_name": method,
            "k": K,
            "popularity_window_days": RECENT_DAYS,
            "repeat_products_allowed": True,
            "inventory_filter": "unavailable_in_olist",
            "validation_snapshot": str(REC_PERIODS["validation"][0].date()),
            "validation_label_end": str(REC_PERIODS["validation"][1].date()),
            "test_snapshot": str(REC_PERIODS["test"][0].date()),
            "test_label_end": str(REC_PERIODS["test"][1].date()),
            "is_champion_baseline": is_champion,
        })
        
        # 2. Log Tags
        mlflow.set_tag("pipeline_integrated", "true")
        mlflow.set_tag("selected_on_validation", str(is_champion))
        if is_champion:
            mlflow.set_tag("model_status", "champion_baseline")
            
        # 3. Log Metrics for validation and test splits
        for split in ["validation", "test"]:
            split_prefix = "val" if split == "validation" else "test"
            row = recommendation_metrics.loc[(recommendation_metrics["baseline"] == method) & (recommendation_metrics["split"] == split)].iloc[0]
            
            mlflow.log_metrics({
                f"{split_prefix}_ndcg_at_k": float(row["ndcg_at_k"]),
                f"{split_prefix}_recall_at_k": float(row["recall_at_k"]),
                f"{split_prefix}_precision_at_k": float(row["precision_at_k"]),
                f"{split_prefix}_hit_rate_at_k": float(row["hit_rate_at_k"]),
                f"{split_prefix}_catalog_coverage": float(row["catalog_coverage"]),
                f"{split_prefix}_customers": int(row["customers"]),
                f"{split_prefix}_warm_customers": int(row["warm_customers"]),
            })
            
            # Log cohort specific NDCG & Recall
            for cohort in ["warm", "new"]:
                cohort_match = recommendation_cohorts.loc[
                    (recommendation_cohorts["baseline"] == method) &
                    (recommendation_cohorts["split"] == split) &
                    (recommendation_cohorts["cohort"] == cohort)
                ]
                if not cohort_match.empty:
                    c_row = cohort_match.iloc[0]
                    mlflow.log_metrics({
                        f"{split_prefix}_{cohort}_ndcg_at_k": float(c_row["ndcg_at_k"]),
                        f"{split_prefix}_{cohort}_recall_at_k": float(c_row["recall_at_k"]),
                    })
                    
        # 4. Log Artifacts
        mlflow.log_artifact(str(BASELINES_DIR / "recommendation_metrics.csv"))
        mlflow.log_artifact(str(BASELINES_DIR / "recommendation_cohorts.csv"))
        mlflow.log_artifact(str(BASELINES_DIR / "example_recommendations.csv"))
        
        mlflow_runs.append({
            "baseline": method,
            "run_id": run.info.run_id,
            "val_ndcg_at_k": float(val_subset.loc[val_subset["baseline"] == method, "ndcg_at_k"].iloc[0]),
            "val_recall_at_k": float(val_subset.loc[val_subset["baseline"] == method, "recall_at_k"].iloc[0]),
            "champion": is_champion
        })

runs_df = pd.DataFrame(mlflow_runs).sort_values("val_ndcg_at_k", ascending=False).reset_index(drop=True)
display(runs_df)
print("\nLogged recommendation baseline runs successfully to MLflow experiment:", EXPERIMENT_NAME)


🏆 Champion Recommendation Baseline selected on Validation NDCG@10: 'purchase_history'
              baseline  ... champion
0     purchase_history  ...     True
1  category_popularity  ...    False
2    global_popularity  ...    False

[3 rows x 5 columns]

Logged recommendation baseline runs successfully to MLflow experiment: ecommerce-recommendations


## 5. Model Analysis & Key Business Insights

1. **The Cold-Start Dominance in E-Commerce**:
   - In Olist, **~97.7% of customers in any given month are brand new buyers** (5,961 out of 6,100 in validation).
   - For these new customers, personalized rules (`purchase_history` and `category_popularity`) have zero past data and fall back to `global_popularity`.
   - As a result, all three baselines achieve identical scores on new customers (**NDCG@10 = 0.00575**).

2. **The Power of Personalization on Repeat Buyers (`warm` cohort)**:
   - For the **139 repeat customers** with prior purchase history, the differences are dramatic:
     - `global_popularity`: NDCG@10 = **0.0044** (worse than random popular items)
     - `category_popularity`: NDCG@10 = **0.0236** (5.3x higher than global)
     - `purchase_history`: NDCG@10 = **0.1214** (27.5x higher than global!)
   - **Conclusion**: Customers who return to the platform have very high loyalty to products they have previously bought or their specific category preferences.

3. **Champion Baseline**:
   - **`purchase_history`** wins overall with validation **NDCG@10 = 0.008388** and **Recall@10 = 0.014590**.
   - Any future personalized collaborative filtering (Matrix Factorization, Two-Tower DSSM) **must beat 0.008388 NDCG@10** on validation to be approved for deployment.


## 6. Pipeline Architecture: Reusable `BaselineRecommender` Class

To prepare for the python recommendation pipeline (`src/models/baseline/recommendation_baselines.py`), we encapsulate this recommendation logic into a reusable Python class.


In [7]:
class BaselineRecommender:
    """Standardized baseline recommender for top-K product ranking.
    
    Supported methods:
      - 'purchase_history': Customer past purchases, fallback to global popularity (Default Champion)
      - 'category_popularity': Customer recent category top items, fallback to global popularity
      - 'global_popularity': Top purchased products in past 90 days
    """
    
    def __init__(self, history: pd.DataFrame, cutoff: pd.Timestamp, method: str = "purchase_history", k: int = 10):
        self.method = method
        self.k = k
        self.cutoff = cutoff
        self._recommender_fn = prepare_recommender(history, cutoff)
        
    def recommend(self, customer_id: str, k: int | None = None) -> list[str]:
        top_k = k or self.k
        return self._recommender_fn(customer_id, self.method, top_k)

# Quick smoke test of the BaselineRecommender class
history_sample = load_history(WAREHOUSE_PATH, pd.Timestamp("2018-07-01"))
rec_model = BaselineRecommender(history_sample, pd.Timestamp("2018-07-01"), method="purchase_history", k=10)

test_cust = history_sample.iloc[0]["customer_unique_id"]
test_recommendations = rec_model.recommend(test_cust, k=5)
print(f"Sample Recommendations for Customer '{test_cust[:8]}...':")
print(test_recommendations)


Sample Recommendations for Customer '871766c5...':
['4244733e06e7ecb4970a6e2683c13e61', '53b36df67ebb7c41585e8d54d6772e08', 'aca2eb7d00ea1a7b8ebd4e68314663af', '3dd2a17168ec895c781a9191c1e95ad7', '422879e10f46682990de24d770e7f83d']


## 7. Summary & Next Steps

1. **Three Recommendation Baselines Built & Evaluated**:
   - `global_popularity`: Ranks products by recent 90-day platform-wide order frequency.
   - `category_popularity`: Explores top items in the customer's recent categories, padded with global items.
   - `purchase_history`: **Champion Baseline** (Validation NDCG@10 = **0.008388**, Warm-Cohort NDCG@10 = **0.121403**).
2. **MLflow Tracking Configured**:
   - Experiments logged under dedicated experiment `ecommerce-recommendations`.
   - Parameters, dual-split metrics, cohort breakdowns (Warm vs. New), and champion tags recorded.
3. **Production Module & CLI Available**:
   - The `BaselineRecommender` class and pipeline are implemented in [`src/models/baseline/recommendation_baselines.py`](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/src/models/baseline/recommendation_baselines.py).
   - CLI launcher available at [`scripts/models/baseline/recommendation_baselines.py`](file:///Users/saiprasadsabeson/Documents/Workspace/iiitb/ecommerce-forecasting-recommender/scripts/models/baseline/recommendation_baselines.py).
